In [ ]:
pip install -U deepeval

In [ ]:
import os
import pandas as pd
import time
import math
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from langchain.chat_models import AzureChatOpenAI
# from langchain_openai import AzureChatOpenAI
from deepeval.models.base_model import DeepEvalBaseLLM
from datetime import datetime

os.environ["OPENAI_API_TYPE"] = "azure"
os.environ["OPENAI_API_VERSION"] = "2024-02-15-preview"
os.environ["AZURE_OPENAI_API_KEY"] = 'xxxx'
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://yyyy.com/"

def round_up(n, decimals = 0):
    multiplier = 10**decimals
    return math.ceil(n * multiplier) / multiplier

class AzureOpenAI(DeepEvalBaseLLM):
    def __init__(
        self,
        model
    ):
        self.model = model

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        return chat_model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = await chat_model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return "Custom Azure OpenAI Model"

custom_model = AzureChatOpenAI(
    openai_api_version= "2024-02-15-preview",
    azure_deployment= "gpt-4-turbo",
    temperature=0,
    model_version="turbo-2024-04-09",
)


azure_openai = AzureOpenAI(model=custom_model)
base_path = '/home/qmed-intel/Desktop/Notebook/Shamus/ipex+bf16'
file_name = 'inference_n=50.csv'
inference_results_path = os.path.join(base_path, file_name)

df = pd.read_csv(inference_results_path)

score_breakdown = []
results = []
reason = []
start_time = 0
end_time = 0
exe_time = 0
total_exe_time = 0
avg_exe_time = 0
query = 0
score = 0
alignment_score = 0
coverage_score = 0

for index, row in df.iterrows():
    query += 1
    print("Query:", query, "\n")
    # print(row['Input_Query'] + "\n")
    # print(row['Output_Response'])
    # This is the original text to be summarized
    input_query = row['Input Query']
    
    # This is the summary, replace this with the actual output from your LLM application
    actual_output = row['Generated Output']
    
    test_case = LLMTestCase(input=input_query, actual_output=actual_output)
    metric = SummarizationMetric(
        threshold=0.5, 
        model=azure_openai, 
        assessment_questions=[
     "Does this summary clearly state the age and gender of the patient?", 
    "Does it cover the main symptom that concerns the patient the most (Chief Complaint) and the duration it has been present?", 
    "Does it cover the elaboration of the main symptom, additional symptoms associated with the chief complaint and important negative history?", 
    "Does the summary include relevant past medical history, such as chronic conditions, allergy history or previous significant illnesses and information on any past surgical procedures the patient has undergone?", 
    "Does the summary include relevant family medical history and social history on smoking and alcohol consumption?"
        ]
    )
    
    start_time = time.time()
    metric.measure(test_case)
    end_time = time.time()

    exe_time = round_up((end_time - start_time), 4)

    total_exe_time += exe_time

    print("Execution time:", exe_time, "\n")
    
    print(metric.score)
    print(metric.reason)
    print(metric.score_breakdown)
    print("\n")
    score_breakdown = metric.score_breakdown
    # if metric.score >= 0.85:
    results.append({
        'Input Query': input_query, 
        'Generated Summary': actual_output, 
        'Score': metric.score, 
        'Reason': metric.reason, 
        'Alignment Score': score_breakdown['Alignment'], 
        'Coverage Score': score_breakdown['Coverage'], 
        'Execution Time (s)': exe_time, 
    })

    time.sleep(3)

avg_exe_time = total_exe_time / query

print("Overall Average Execution Time:", avg_exe_time)


df = pd.DataFrame(results)
current_datetime = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# Save the DataFrame to a CSV file
base_path = '/home/qmed-intel/Desktop/Notebook/Shamus/ipex+bf16'
file_name = 'deepeval_n=50.csv'
deepeval_path = os.path.join(base_path, file_name)
os.makedirs(base_path, exist_ok=True)


df.to_csv(deepeval_path, index=False)
print(f"Results successfully saved to {deepeval_path}")